# 03 — Multimodal RAG

**Priority:** 🟢 Nice-to-have — valuable breadth for image+text corpora specifically. *If skipped, revisit when:* when a corpus includes images/diagrams.

```
╔═══════════════════════════════════════════════════════════════════════╗
║                    3. MULTIMODAL RAG                                  ║
║                                                                       ║
║  Multimodal Docs ──► Multimodal Embedding Model ──► Vector DB        ║
║  (Text, Images,                     ▲              (shared space)    ║
║   Tables, Audio)                    │                    │           ║
║                                     │                    │           ║
║  Query ─────────────────────────────┘              top-k │           ║
║                                                          ▼           ║
║  Response ◄── Multimodal Generative Model ◄── Prompt + Media Assets  ║
╚═══════════════════════════════════════════════════════════════════════╝
```

## What is Multimodal RAG?

Standard RAG only retrieves text. Multimodal RAG extends this to **images** (and potentially audio, video, tables) by using a **shared embedding space** that maps both text and images to the same vector dimensions.

The key ingredient is **CLIP** (Contrastive Language–Image Pre-training), which was trained to make `embed("a cat")` close to `embed(image_of_cat)` in the same vector space.

This enables:
- **Text query → image results**: "What does the HeliosArm joint diagram look like?"
- **Image query → text results**: Upload a photo of a connector, find related documentation
- **Mixed results**: Return both text chunks AND relevant images

## What you'll learn
- Embed images and text into a shared CLIP space
- Build a unified multimodal index
- Do text→image and image→text retrieval
- Pass retrieved images to a vision LLM for answering

In [ ]:
import sys; sys.path.insert(0, '..')
import ragkit.config as cfg

cfg.BACKEND = "claude"   # "claude" | "local"
print(f"Backend: {cfg.BACKEND}  |  Device: {cfg.DEVICE}")

## Step 1 — Understand CLIP's shared embedding space

CLIP was trained with pairs of (image, caption) to maximise the similarity between matching pairs and minimise it for non-matching pairs. As a result, semantically related text and images cluster together in the same vector space.

> **A note on CLIP's age:** `clip-ViT-B-32` (used below) is from 2021 — small (~300MB), text-image only, and noticeably weaker at text-only retrieval than a dedicated text embedder (see the Tradeoffs table at the end of this notebook). It stays the default here because it's the only multimodal option that runs fully offline with no API key — exactly what's needed to *learn* the shared-embedding-space concept.
>
> As of mid-2026, production multimodal RAG reaches for newer models instead: **Google's Gemini Embedding** (natively multimodal — text, image, video, and audio in one shared space, not just text-image pairs like CLIP), **Cohere embed-v4** (multimodal, variable output dimensions), or **Voyage's multimodal embeddings** (from Anthropic's recommended embedding partner). These generally beat CLIP on both image retrieval *and* the text-retrieval weakness called out below.

In [ ]:
from ragkit.embeddings import clip_embed_texts, clip_embed_images
from ragkit.data import load_images
import numpy as np

# Load our 4 Helios images
images = load_images()
print(f"Loaded {len(images)} images")
for img in images:
    print(f"  {img['filename']}")
    print(f"    Caption: {img['caption'][:80]}")

In [ ]:
# Embed both images and some text queries into the SAME vector space
text_probes = [
    "robotic arm joint diagram",
    "mobile robot top view with safety zones",
    "battery capacity degradation chart",
    "controller panel with ports",
    "weather forecast",  # should be far from all images
]

print("Embedding images with CLIP (first call downloads the model ~300MB)...")
image_vecs = clip_embed_images([img['path'] for img in images])
text_vecs  = clip_embed_texts(text_probes)

print(f"Image embeddings: {image_vecs.shape}")
print(f"Text embeddings:  {text_vecs.shape}")
print(f"Same dimension: {image_vecs.shape[1] == text_vecs.shape[1]} ✓")

In [ ]:
# Compute text-to-image similarity matrix
import matplotlib.pyplot as plt

sim_matrix = text_vecs @ image_vecs.T  # already normalised → cosine similarity

fig, ax = plt.subplots(figsize=(8, 5))
im = ax.imshow(sim_matrix, cmap='RdYlGn', vmin=-0.1, vmax=0.4)
ax.set_xticks(range(len(images)))
ax.set_xticklabels([img['filename'].replace('spec_','').replace('.png','')[:18]
                    for img in images], rotation=30, ha='right', fontsize=8)
ax.set_yticks(range(len(text_probes)))
ax.set_yticklabels(text_probes, fontsize=8)
plt.colorbar(im, ax=ax, label='Cosine similarity')
ax.set_title('CLIP Text–Image Similarity Matrix', fontsize=11)

for i in range(len(text_probes)):
    for j in range(len(images)):
        ax.text(j, i, f"{sim_matrix[i,j]:.2f}", ha='center', va='center', fontsize=7)
        
plt.tight_layout(); plt.show()
print("\n→ Diagonal (matching pairs) should have the highest values.")
print("→ 'weather forecast' should have near-zero similarity with all images.")

## Step 2 — Build the unified multimodal index

We index **both** text chunks and images into the same Chroma collection, using CLIP embeddings. A `type` metadata field distinguishes them.

In [ ]:
import chromadb
from ragkit.data import load_corpus, chunk_text

client = chromadb.PersistentClient(path="../.chroma")
try:
    client.delete_collection("helios_multimodal")
except Exception:
    pass
mm_collection = client.create_collection("helios_multimodal", metadata={"hnsw:space": "cosine"})

# ── Index text chunks via CLIP text encoder ───────────────────────────────────
all_texts, all_metas, all_ids, all_vecs = [], [], [], []

docs = load_corpus()
for doc in docs:
    chunks = chunk_text(doc['text'], chunk_size=120, overlap=30)
    for i, chunk in enumerate(chunks):
        all_texts.append(chunk)
        all_metas.append({'type': 'text', 'source': doc['source'], 'category': doc['category'], 'chunk': i})
        all_ids.append(f"text_{doc['source']}_{i}")

print(f"Text chunks to index: {len(all_texts)}")
text_clip_vecs = clip_embed_texts(all_texts)
print(f"Text CLIP embeddings: {text_clip_vecs.shape}")

# ── Index images via CLIP image encoder ──────────────────────────────────────
img_texts, img_metas, img_ids = [], [], []
for img in images:
    img_texts.append(img['caption'])  # text representation stored in collection
    img_metas.append({'type': 'image', 'path': img['path'], 'filename': img['filename'],
                      'category': img['category'], 'caption': img['caption']})
    img_ids.append(f"img_{img['filename']}")

img_clip_vecs = clip_embed_images([img['path'] for img in images])
print(f"Image CLIP embeddings: {img_clip_vecs.shape}")

# ── Add everything to one collection ─────────────────────────────────────────
import numpy as np

mm_collection.add(
    ids=all_ids + img_ids,
    documents=all_texts + img_texts,
    embeddings=np.vstack([text_clip_vecs, img_clip_vecs]).tolist(),
    metadatas=all_metas + img_metas,
)
print(f"\nMultimodal collection: {mm_collection.count()} items ({len(all_texts)} text + {len(images)} images)")

## Step 3 — Text query → image and text results

In [ ]:
from PIL import Image as PILImage

def multimodal_retrieve(query_text: str, collection, k: int = 5):
    """Retrieve text and image results from the multimodal collection."""
    q_vec = clip_embed_texts([query_text]).tolist()
    results = collection.query(
        query_embeddings=q_vec,
        n_results=min(k, collection.count()),
        include=['documents', 'metadatas', 'distances']
    )
    
    hits = []
    for doc, meta, dist in zip(results['documents'][0], results['metadatas'][0], results['distances'][0]):
        hits.append({'text': doc, 'metadata': meta, 'score': 1 - dist})
    return hits

def display_results(hits, query_text):
    """Display mixed text+image results."""
    text_hits = [h for h in hits if h['metadata']['type'] == 'text']
    img_hits  = [h for h in hits if h['metadata']['type'] == 'image']
    
    print(f"Query: '{query_text}'")
    print(f"Results: {len(text_hits)} text chunks, {len(img_hits)} images")
    
    if text_hits:
        print("\nText results:")
        for i, h in enumerate(text_hits[:3]):
            print(f"  [{h['score']:.3f}] [{h['metadata']['source']}] {h['text'][:120]}...")
    
    if img_hits:
        print(f"\nImage results:")
        for h in img_hits:
            print(f"  [{h['score']:.3f}] {h['metadata']['filename']}")
            print(f"           {h['metadata']['caption'][:100]}")
        
        fig, axes = plt.subplots(1, len(img_hits), figsize=(5*len(img_hits), 3))
        if len(img_hits) == 1:
            axes = [axes]
        for ax, h in zip(axes, img_hits):
            img = PILImage.open(h['metadata']['path'])
            ax.imshow(img)
            ax.set_title(f"score={h['score']:.3f}\n{h['metadata']['filename'][:30]}", fontsize=7)
            ax.axis('off')
        plt.tight_layout(); plt.show()
    
    return text_hits, img_hits

# Test text→image retrieval
q1 = "show me the safety zones around the mobile robot"
hits1 = multimodal_retrieve(q1, mm_collection, k=6)
text_h, img_h = display_results(hits1, q1)

In [ ]:
q2 = "joint diagram for the robotic arm"
hits2 = multimodal_retrieve(q2, mm_collection, k=6)
text_h2, img_h2 = display_results(hits2, q2)

## Step 4 — Pass retrieved images to a vision LLM

When images are retrieved, we can pass them directly to Claude's vision API (or `llama3.2-vision` locally) for image-grounded answers.

In [ ]:
from ragkit.llm import generate
from ragkit.pretty import show_answer

VISION_SYSTEM = """You are a technical support assistant for Helios Robotics.
You will be shown images and/or text context from our knowledge base.
Answer the user's question based on what you can see in the images and the context.
Be specific about what you observe in the images — describe measurements, labels, colours, zones shown."""

def multimodal_rag(question: str, collection, k: int = 6) -> str:
    # Retrieve mixed results
    hits = multimodal_retrieve(question, collection, k=k)
    text_hits = [h for h in hits if h['metadata']['type'] == 'text']
    img_hits  = [h for h in hits if h['metadata']['type'] == 'image']
    
    # Build text context
    text_ctx = "\n\n".join(
        f"[{h['metadata']['source']}]\n{h['text']}"
        for h in text_hits[:3]
    )
    
    # Build prompt
    image_note = ""
    if img_hits:
        captions = "; ".join(h['metadata']['caption'][:60] for h in img_hits)
        image_note = f"\nThe attached image(s) show: {captions}"
    
    prompt = f"""Context from knowledge base:{image_note}

{text_ctx}

---
Question: {question}"""
    
    # Pass images if any were retrieved
    image_paths = [h['metadata']['path'] for h in img_hits] if img_hits else None
    
    return generate(prompt, system=VISION_SYSTEM, images=image_paths)

# Image-grounded question
q_img = "Looking at the HeliosBase M1 diagram, what are the two safety zones shown and what are their dimensions?"
answer = multimodal_rag(q_img, mm_collection)
show_answer(answer, title=f"Multimodal RAG — '{q_img[:50]}...'")

In [ ]:
# Purely text query — should still work (no images retrieved)
q_text = "What is the rated payload of the HeliosArm V2?"
answer2 = multimodal_rag(q_text, mm_collection)
show_answer(answer2, title=f"Multimodal RAG (text-only) — '{q_text}'")

In [ ]:
# Technical diagram question — combines image + text evidence
q_chart = "Based on the battery capacity chart, at what cycle count should I be worried about unexpected shutdowns?"
answer3 = multimodal_rag(q_chart, mm_collection)
show_answer(answer3)

## Step 5 — Image query → text retrieval (reverse direction)

We can also use an image as the query to find related text — like reverse image search for documentation.

In [ ]:
# Use the arm joint diagram as a query to find related text
arm_image_path = images[0]['path']  # spec_arm_v2_joint_diagram.png
print(f"Query image: {images[0]['filename']}")

# Embed the image with CLIP
img_q_vec = clip_embed_images([arm_image_path]).tolist()

# Search against the text-only part of the collection
results = mm_collection.query(
    query_embeddings=img_q_vec,
    n_results=5,
    where={"type": "text"},  # filter to text only
    include=['documents', 'metadatas', 'distances']
)

print("\nTop text documents retrieved by the arm diagram image:")
for i, (doc, meta, dist) in enumerate(zip(
    results['documents'][0], results['metadatas'][0], results['distances'][0]
)):
    print(f"\n  {i+1}. [score={1-dist:.3f}] {meta['source']}")
    print(f"     {doc[:150]}...")

## Exercise — Text and images in one shared space (Drag Race runways)

CLIP puts **text** and **images** into the *same* vector space, so a text query can be scored directly against an image. This exercise mimics that with a hand-built space describing runway looks over the dimensions **[red, sequins, feathers, long-silhouette]**.

1. **Predict first**: For the query *"a sparkly sequined outfit"*, which runway image should win? And for *"a dramatic feathered look"*?
2. **Implement**: Score each text query against every image with cosine similarity and pick the best image (cross-modal retrieval).
3. **Reflect**: One sentence — why does a *shared* space let you skip generating a caption for each image first?

In [ ]:
import numpy as np

def cosine_sim(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

# Runway "images" embedded in a shared space.  dims: [red, sequins, feathers, long_silhouette]
images = {
    "img_red_gown":        [0.90, 0.30, 0.10, 0.80],
    "img_feather_boa":     [0.20, 0.20, 0.95, 0.40],
    "img_sequin_bodysuit": [0.30, 0.95, 0.20, 0.30],
    "img_long_cape":       [0.40, 0.30, 0.20, 0.95],
}

# Text queries embedded in the SAME space.
text_queries = {
    "a sparkly sequined outfit": [0.20, 0.95, 0.10, 0.20],
    "a dramatic feathered look": [0.10, 0.20, 0.95, 0.30],
}

# ── Task 1: predictions (write before running) ────────────────────────────────
#   sparkly sequined  -> ____________
#   dramatic feathered -> ____________

# ── Task 2: cross-modal retrieval (text query -> best image) ──────────────────
best = {}
for q, qv in text_queries.items():
    scores = {im: cosine_sim(qv, v) for im, v in images.items()}
    best[q] = max(scores, key=scores.get)
    ranked = sorted(scores.items(), key=lambda kv: kv[1], reverse=True)
    print(f"{q!r}")
    for im, s in ranked:
        print(f"    {s:.3f}  {im}")
    print(f"   → best: {best[q]}\n")

# ── Task 3: why a shared space helps (comment) ────────────────────────────────
#   Your answer:

# ── Self-check ────────────────────────────────────────────────────────────────
assert best["a sparkly sequined outfit"] == "img_sequin_bodysuit"
assert best["a dramatic feathered look"] == "img_feather_boa"
print("✅ Exercise checks passed!")

## Tradeoffs

| Aspect | Naive RAG | Multimodal RAG |
|---|---|---|
| **Data types** | Text only | Text + Images (+Audio/Video with more models) |
| **Embedding model** | Sentence-transformers | CLIP (2021, ~300MB) here for offline learning — production uses Gemini Embedding, Cohere embed-v4, or Voyage multimodal instead |
| **Retrieval quality (text)** | ★★★★☆ | ★★★☆☆ (CLIP is optimised for text-image, not text-text — modern multimodal embedders close most of this gap) |
| **Image retrieval** | ✗ Not possible | ✓ |
| **LLM requirements** | Any text LLM | Vision LLM needed for image-grounded answers |
| **Index size** | Small | Larger (image embeddings same size as text) |
| **When to use** | Text-only corpora | Corpora with diagrams, photos, charts, mixed media |

**Practical tip**: In production, it's common to use CLIP for image retrieval but sentence-transformers for text retrieval, then merge the result sets (heterogeneous retrieval).

## Exercises

1. **Upload your own image**: Take a photo of a robot (any robot) and use it as a query against the collection. What text does it retrieve?
2. **CLIP limitations**: Try a very specific technical question like "what is the torque spec for Joint 4?". Does the CLIP-indexed text collection answer it as well as the sentence-transformer index from notebook 01?
3. **Hybrid text indexing**: Build two separate collections — one with CLIP embeddings, one with sentence-transformers — and merge their results for text queries. Does this help?
4. **Add a PDF**: Find any technical manual PDF, extract some pages as images, and add them to the multimodal collection.

**Next:** [04_graph_rag.ipynb](04_graph_rag.ipynb) — tackle multi-hop questions by building a real knowledge graph.